In [3]:
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
import numpy as np

print(f"ChromaDB version: {chromadb.__version__}")
print("Day 12 - Chroma Deep Dive")

# Initialize embedding model
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Embedded loaded")

ChromaDB version: 1.5.9
Day 12 - Chroma Deep Dive
Embedded loaded


In [4]:
import os

print("=== ChromaDB Persistent Client ===\n")

# Persistent client - data survives kernel restartd
# In production this points to a Docker volume
client = chromadb.PersistentClient(path="./chroma_db")

print(f"Client created")
print(f"Storage path: ./chroma_db")

# List existing collection
collection = client.get_or_create_collection(
    name="enterprise_rag",
    metadata={
        "description": "Main RAG document collection",
        "embedding_model": "all-MiniLM-L6-v2",
        "created_by": "jay_123"
    }
)

print(f"\nCollection: {collection.name}")
print(f"Collection metadata:{collection.metadata}")
print(f"Document in collection: {collection.count()}")

=== ChromaDB Persistent Client ===

Client created
Storage path: ./chroma_db

Collection: enterprise_rag
Collection metadata:{'created_by': 'jay_123', 'embedding_model': 'all-MiniLM-L6-v2', 'description': 'Main RAG document collection'}
Document in collection: 8


In [11]:
print("=== Adding Documents with Metadata ===\n")

# Simulate documents from different users and sources
documents = [
    {
        "id": "doc_001",
        "text": "Hybrid search combines BM25 keyword search with vector semantic search for better retrieval.",
        "metadata": {"source": "rag_paper.pdf", "page": 1, "author": "jay", "org":"acme_corp", "topic": "retrieval"}
    },
    {
        "id": "doc_002",
        "text": "Reciprocal Rank Fusion merges results from multiple search methods using rank position.",
        "metadata": {"source": "rag_paper.pdf", "page":2, "author": "jay", "org": "acme_corp", "topic":"retrieval"}
    },
    {
        "id": "doc_003",
        "text":"RAGAs evaluates faithfullness by checking if answers are grounded in retrieved context.",
        "metadata":{"source": "eval_guide.pdf", "page": 1, "author":"alice", "org": "acme_corp", "topic": "evaluation"}
    },
    {
        "id": "doc_004",
        "text":"Cross-encoders re-rankers process query and document together for higher precision scoring.",
        "metadata":{"source": "reranking.pdf", "page": 1, "author":"bob", "org": "techco", "topic": "retrieval"}
    },
    {
        "id": "doc_005",
        "text":"LoRA fine-tuning reduces trainable parameters by 90 percent using low-rank matrix decomposition.",
        "metadata":{"source": "finetuning.pdf", "page": 1, "author":"bob", "org": "techco", "topic": "training"}
    },
    {
        "id": "doc_006",
        "text":"FastAPI provides automatic OpenAPI documentation and sync request handling for ML backends.",
        "metadata":{"source": "deployment.pdf", "page": 1, "author":"carol", "org": "acme_corp", "topic": "deployment"}
    },
    {
        "id": "doc_007",
        "text":"Docker containerization ensures consistent environments across development and production.",
        "metadata":{"source": "deployment.pdf", "page": 2, "author":"carol", "org": "acme_corp", "topic": "deployment"}
    },
    {
        "id": "doc_008",
        "text":"ChromaDB supports multi-tenant namespacing through collection isolation per organisation.",
        "metadata":{"source": "chromadb.pdf", "page": 1, "author":"jay", "org": "acme_corp", "topic": "database"}
    }    
]

# Generate embeddings for all documents
texts = [doc["text"] for doc in documents]
embeddings= embedder.encode(texts).tolist()

# Add to ChromaDB
collection.add(
    ids=[doc["id"] for doc in documents],
    embeddings=embeddings,
    documents=texts,
    metadatas=[doc["metadata"] for doc in documents]

)

print(f"Added {len(documents)} documents")
print(f"Total in collection: {collection.count()}")

# Verify one document
result = collection.get(ids=["doc_001"], include=["metadatas", "documents", "embeddings"])
print(f"\nVerification - doc_001:")
print(f"  embeddings: {result['embeddings'][0][:5]}")
print(f"  Text: {result['documents'][0][:60]}...")
print(f"  Metadata {result['metadatas'][0]}")

=== Adding Documents with Metadata ===

Added 8 documents
Total in collection: 8

Verification - doc_001:
  embeddings: [ 0.01681826  0.00618585 -0.01113682 -0.0162531   0.02966382]
  Text: Hybrid search combines BM25 keyword search with vector seman...
  Metadata {'source': 'rag_paper.pdf', 'topic': 'retrieval', 'page': 1, 'org': 'acme_corp', 'author': 'jay'}


In [6]:
print("=== Similarity Search ===\n")

# Basic similarity search
query = "how does retrieval work in RAG?"
query_embedding = embedder.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print(f"Query: '{query}'\n")
print("Top 3 results:")
for i in range(len(results['ids'][0])):
    doc_id = results['ids'][0][i]
    text = results['documents'][0][i]
    distance = results['distances'][0][i]
    metadata = results['metadatas'][0][i]
    print(f"\n  Rank {i+1} | ID: {doc_id} | Distance: {distance:.4f}")
    print(f"  Text: {text[:70]}...")
    print(f"  Source: {metadata['source']} | Org: {metadata['org']}")

print("\n=== Metadata Filtering ===\n")

# Filter by organization - multi-tenancy in action
print("Query restricted to acme_corp only:")
acme_results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    where={"org": "acme_corp"} # only returns acme_corp docs
)

for i in range(len(acme_results['ids'][0])):
    doc_id = acme_results['ids'][0][i]
    text = acme_results['documents'][0][i]
    org= acme_results['metadatas'][0][i]['org']
    print(f" [{org}] {doc_id}: {text[:60]}...")

print("\n Query restricted to techo only:")
techo_results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    where={"org": "techco"} # only returns techo results
)

for i in range(len(techo_results['ids'][0])):
    doc_id=techo_results['ids'][0][i]
    text= techo_results['documents'][0][i]
    org = techo_results['metadatas'][0][i]['org']
    print(f"  [{org}] {doc_id}: {text[:60]}...")

=== Similarity Search ===

Query: 'how does retrieval work in RAG?'

Top 3 results:

  Rank 1 | ID: doc_003 | Distance: 1.0578
  Text: RAGAs evaluates faithfullness by checking if answers are grounded in r...
  Source: eval_guide.pdf | Org: acme_corp

  Rank 2 | ID: doc_002 | Distance: 1.6171
  Text: Reciprocal Rank Fusion merges results from multiple search methods usi...
  Source: rag_paper.pdf | Org: acme_corp

  Rank 3 | ID: doc_004 | Distance: 1.6409
  Text: Cross-encoder re-rankers process query and document together for highe...
  Source: reranking.pdf | Org: techco

=== Metadata Filtering ===

Query restricted to acme_corp only:
 [acme_corp] doc_003: RAGAs evaluates faithfullness by checking if answers are gro...
 [acme_corp] doc_002: Reciprocal Rank Fusion merges results from multiple search m...
 [acme_corp] doc_001: Hybrid search combines BM25 keyword search with vector seman...

 Query restricted to techo only:
  [techco] doc_004: Cross-encoder re-rankers process query and 

In [25]:
all_docs = collection.get()
for meta in all_docs['metadatas']:
    if meta['author'] == 'bob':
        print(meta)

{'author': 'bob', 'org': 'techco', 'source': 'reranking.pdf', 'topic': 'retrieval', 'page': 1}
{'source': 'finetuning.pdf', 'author': 'bob', 'org': 'techco', 'page': 1, 'topic': 'training'}


In [12]:
print("=== Advanced Metadata Filtering ===\n")

# Filter by topic
print("=== Topic: retrieval only ===")
retrieval_results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5,
    where={"topic": "retrieval"}
)
for i in range(len(retrieval_results['ids'][0])):
    meta = retrieval_results['metadatas'][0][i]
    text = retrieval_results['documents'][0][i]
    print(f"  [{meta['topic']}] {text[:60]}...")

# Filter by author
print("\n=== Author: jay only ===")
jay_results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5,
    where={"author": "jay"}
)
for i in range(len(jay_results['ids'][0])):
    meta = jay_results['metadatas'][0][i]
    text = jay_results['documents'][0][i]
    print(f"  [{meta['author']}] {text[:60]}...")

# Compound filter — acme_corp AND retrieval topic
print("\n=== acme_corp AND retrieval topic ===")
compound_results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5,
    where={
        "$and": [
            {"org": "acme_corp"},
            {"topic": "retrieval"}
        ]
    }
)
for i in range(len(compound_results['ids'][0])):
    meta = compound_results['metadatas'][0][i]
    text = compound_results['documents'][0][i]
    print(f"  [{meta['org']}|{meta['topic']}] {text[:60]}...")

# Get all documents from a specific source
print("\n=== All docs from rag_paper.pdf ===")
source_results = collection.get(
    where={"source": "rag_paper.pdf"}
)
for i in range(len(source_results['ids'])):
    print(f"  {source_results['ids'][i]}: {source_results['documents'][i][:60]}...")

=== Advanced Metadata Filtering ===

=== Topic: retrieval only ===
  [retrieval] Reciprocal Rank Fusion merges results from multiple search m...
  [retrieval] Cross-encoder re-rankers process query and document together...
  [retrieval] Hybrid search combines BM25 keyword search with vector seman...

=== Author: jay only ===
  [jay] Reciprocal Rank Fusion merges results from multiple search m...
  [jay] Hybrid search combines BM25 keyword search with vector seman...
  [jay] ChromaDB supports multi-tenant namespacing through collectio...

=== acme_corp AND retrieval topic ===
  [acme_corp|retrieval] Reciprocal Rank Fusion merges results from multiple search m...
  [acme_corp|retrieval] Hybrid search combines BM25 keyword search with vector seman...

=== All docs from rag_paper.pdf ===
  doc_001: Hybrid search combines BM25 keyword search with vector seman...
  doc_002: Reciprocal Rank Fusion merges results from multiple search m...


In [18]:
from typing import List, Dict, Optional
import uuid

class MultiTenantVectorStore:
    """
    Manages ChromaDB collections with multi-tenant isolation.
    Each org gets its own collection — complete data separation.
    This is what your FastAPI backend will use in Week 5.
    """
    
    def __init__(self, persist_path: str = "./chroma_db"):
        self.client = chromadb.PersistentClient(path=persist_path)
        self.embedder = SentenceTransformer("all-MiniLM-L6-v2")
        self._collections = {}
    
    def _get_collection_name(self, org_id: str) -> str:
        """Each org gets its own collection — complete isolation"""
        return f"org_{org_id}"
    
    def get_or_create_collection(self, org_id: str):
        """Get or create a collection for an organization"""
        collection_name = self._get_collection_name(org_id)
        if collection_name not in self._collections:
            self._collections[collection_name] = self.client.get_or_create_collection(
                name=collection_name,
                metadata={"org_id": org_id}
            )
        return self._collections[collection_name]
    
    def add_documents(
        self,
        org_id: str,
        texts: List[str],
        metadatas: List[Dict],
        ids: Optional[List[str]] = None
    ) -> List[str]:
        """Add documents to an org's collection"""
        collection = self.get_or_create_collection(org_id)
        
        # Auto generate IDs if not provided
        if ids is None:
            ids = [str(uuid.uuid4())[:8] for _ in texts]
        
        # Generate embeddings
        embeddings = self.embedder.encode(texts).tolist()
        
        collection.add(
            ids=ids,
            embeddings=embeddings,
            documents=texts,
            metadatas=metadatas
        )
        
        print(f"[{org_id}] Added {len(texts)} documents")
        return ids
    
    def query(
        self,
        org_id: str,
        query_text: str,
        n_results: int = 3,
        filters: Optional[Dict] = None
    ) -> List[Dict]:
        """Query an org's collection — cannot access other orgs"""
        collection = self.get_or_create_collection(org_id)
        query_embedding = self.embedder.encode(query_text).tolist()
        
        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=min(n_results, collection.count()),
            where=filters
        )
        
        # Format results
        formatted = []
        for i in range(len(results['ids'][0])):
            formatted.append({
                "id": results['ids'][0][i],
                "text": results['documents'][0][i],
                "distance": results['distances'][0][i],
                "metadata": results['metadatas'][0][i]
            })
        
        return formatted
    
    def get_stats(self, org_id: str) -> Dict:
        """Get collection statistics for an org"""
        collection = self.get_or_create_collection(org_id)
        return {
            "org_id": org_id,
            "total_documents": collection.count(),
            "collection_name": collection.name
        }
    
    def __str__(self) -> str:
        return f"MultiTenantVectorStore({len(self._collections)} active collections)"


# Test it
print("=== Multi-Tenant Vector Store ===\n")
store = MultiTenantVectorStore(persist_path="./chroma_db_mt")

# Add documents for org 1
store.add_documents(
    org_id="acme_corp",
    texts=[
        "ACME Corp uses hybrid search for internal document retrieval",
        "ACME Corp financial reports are stored in the RAG system",
        "ACME Corp legal contracts are indexed for quick retrieval"
    ],
    metadatas=[
        {"source": "internal.pdf", "topic": "retrieval"},
        {"source": "finance.pdf", "topic": "finance"},
        {"source": "legal.pdf", "topic": "legal"}
    ]
)

# Add documents for org 2
store.add_documents(
    org_id="techco",
    texts=[
        "TechCo uses RAG for customer support automation",
        "TechCo product documentation is indexed in ChromaDB",
    ],
    metadatas=[
        {"source": "support.pdf", "topic": "support"},
        {"source": "products.pdf", "topic": "products"}
    ]
)

# Query each org separately
print("\n=== Querying acme_corp ===")
acme_results = store.query("acme_corp", "document retrieval system")
for r in acme_results:
    print(f"  [{r['distance']:.3f}] {r['text'][:60]}...")

print("\n=== Querying techco ===")
techco_results = store.query("techco", "document retrieval system")
for r in techco_results:
    print(f"  [{r['distance']:.3f}] {r['text'][:60]}...")

# Stats
print(f"\n=== Stats ===")
print(store.get_stats("acme_corp"))
print(store.get_stats("techco"))
print(f"\n{store}")

=== Multi-Tenant Vector Store ===

[acme_corp] Added 3 documents
[techco] Added 2 documents

=== Querying acme_corp ===
  [0.810] ACME Corp uses hybrid search for internal document retrieval...
  [0.810] ACME Corp uses hybrid search for internal document retrieval...
  [1.257] ACME Corp legal contracts are indexed for quick retrieval...

=== Querying techco ===
  [1.419] Techco product documentation is indexed in ChromaDB...
  [1.419] TechCo product documentation is indexed in ChromaDB...
  [1.620] TechCo uses RAG for customer support automation...

=== Stats ===
{'org_id': 'acme_corp', 'total_documents': 6, 'collection_name': 'org_acme_corp'}
{'org_id': 'techco', 'total_documents': 4, 'collection_name': 'org_techco'}

MultiTenantVectorStore(2 active collections)
